# 05 — Geographic & OS Performance Analysis

Goal: understand how creative performance varies by country and operating system, and identify where budgets are most efficiently deployed.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("../")

daily = pd.read_csv(DATA / "creative_daily_country_os_stats.csv", parse_dates=["date"])
cs = pd.read_csv(DATA / "creative_summary.csv")
camps = pd.read_csv(DATA / "campaigns.csv")

# Compute row-level CTR, CVR, ROAS
daily["ctr"] = daily["clicks"] / daily["impressions"].replace(0, np.nan)
daily["cvr"] = daily["conversions"] / daily["clicks"].replace(0, np.nan)
daily["roas"] = daily["revenue_usd"] / daily["spend_usd"].replace(0, np.nan)

print(f"Daily rows: {len(daily):,}")
print(f"Countries:  {daily['country'].nunique()}")
print(f"OS values:  {daily['os'].nunique()} — {daily['os'].unique()}")

## 1. Top Countries by Impressions and Spend

In [ ]:
country_agg = (
    daily.groupby("country")
    .agg(
        total_impressions=("impressions", "sum"),
        total_spend=("spend_usd", "sum"),
        total_clicks=("clicks", "sum"),
        total_conversions=("conversions", "sum"),
        total_revenue=("revenue_usd", "sum"),
    )
    .reset_index()
)

country_agg["ctr"] = country_agg["total_clicks"] / country_agg["total_impressions"]
country_agg["cvr"] = country_agg["total_conversions"] / country_agg["total_clicks"]
country_agg["roas"] = country_agg["total_revenue"] / country_agg["total_spend"]

top15_impr = country_agg.nlargest(15, "total_impressions")
top15_spend = country_agg.nlargest(15, "total_spend")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].barh(
    top15_impr["country"][::-1],
    top15_impr["total_impressions"][::-1] / 1e6,
    color="#4C72B0",
    edgecolor="white",
)
axes[0].set_title("Top 15 Countries by Total Impressions", fontweight="bold")
axes[0].set_xlabel("Impressions (millions)")

axes[1].barh(
    top15_spend["country"][::-1],
    top15_spend["total_spend"][::-1] / 1e3,
    color="#DD8452",
    edgecolor="white",
)
axes[1].set_title("Top 15 Countries by Total Spend", fontweight="bold")
axes[1].set_xlabel("Spend (USD thousands)")

plt.suptitle("Geographic Distribution of Ad Delivery", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Spend Concentration: What % of Spend Do Top 5 Countries Capture?

In [ ]:
spend_sorted = country_agg.sort_values("total_spend", ascending=False)
top5_share = spend_sorted.head(5)["total_spend"].sum() / spend_sorted["total_spend"].sum() * 100
top10_share = spend_sorted.head(10)["total_spend"].sum() / spend_sorted["total_spend"].sum() * 100

print(f"Top 5 countries capture:  {top5_share:.1f}% of total spend")
print(f"Top 10 countries capture: {top10_share:.1f}% of total spend")
print("\nTop 5 countries by spend:")
print(
    spend_sorted.head(5)[["country", "total_spend", "ctr", "cvr", "roas"]]
    .round(4)
    .to_string(index=False)
)

## 3. CTR and CVR by Country (top 20)

In [ ]:
# Filter to countries with enough volume (>100K impressions)
country_filtered = country_agg[country_agg["total_impressions"] > 100_000]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, col, title, color in [
    (axes[0], "ctr", "Overall CTR by Country (top volume)", "#4C72B0"),
    (axes[1], "cvr", "Overall CVR by Country (top volume)", "#55A868"),
]:
    sorted_data = country_filtered.sort_values(col, ascending=True).tail(20)
    ax.barh(sorted_data["country"], sorted_data[col], color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(col.upper())

plt.suptitle("Country-Level Performance Metrics", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. iOS vs Android — CTR, CVR, ROAS Comparison

In [ ]:
os_agg = (
    daily.groupby("os")
    .agg(
        total_impressions=("impressions", "sum"),
        total_spend=("spend_usd", "sum"),
        total_clicks=("clicks", "sum"),
        total_conversions=("conversions", "sum"),
        total_revenue=("revenue_usd", "sum"),
    )
    .reset_index()
)
os_agg["ctr"] = os_agg["total_clicks"] / os_agg["total_impressions"]
os_agg["cvr"] = os_agg["total_conversions"] / os_agg["total_clicks"]
os_agg["roas"] = os_agg["total_revenue"] / os_agg["total_spend"]

print("OS-level aggregated metrics:")
print(
    os_agg[["os", "total_impressions", "total_spend", "ctr", "cvr", "roas"]]
    .round(4)
    .to_string(index=False)
)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
os_colors = {"Android": "#3ddc84", "iOS": "#555555"}

for ax, metric, label in [
    (axes[0], "ctr", "CTR"),
    (axes[1], "cvr", "CVR"),
    (axes[2], "roas", "ROAS"),
]:
    bars = ax.bar(
        os_agg["os"],
        os_agg[metric],
        color=[os_colors.get(o, "gray") for o in os_agg["os"]],
        edgecolor="white",
        width=0.5,
    )
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.01,
            f"{bar.get_height():.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title(f"{label} by OS", fontweight="bold")
    ax.set_ylabel(label)

plt.suptitle("iOS vs Android Performance Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Country × OS Heatmap (median CTR)

In [ ]:
# Use top 15 countries by volume
top15_countries = country_agg.nlargest(15, "total_impressions")["country"].tolist()
daily_top = daily[daily["country"].isin(top15_countries)]

pivot = daily_top.groupby(["country", "os"])["ctr"].median().unstack("os")

fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".4f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Median Daily CTR"},
)
ax.set_title(
    "Country × OS Heatmap — Median Daily CTR\n(top 15 countries by impressions)", fontweight="bold"
)
ax.set_xlabel("OS")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

## 6. Target OS Campaign Setting vs Actual Performance

In [ ]:
# Join campaign target_os to daily data via campaign_id
daily_with_camp = daily.merge(
    camps[["campaign_id", "target_os", "vertical"]], on="campaign_id", how="left"
)

target_os_perf = (
    daily_with_camp.groupby("target_os")
    .agg(
        median_ctr=("ctr", "median"), median_cvr=("cvr", "median"), median_roas=("roas", "median")
    )
    .reset_index()
)

print("Performance by campaign target_os setting:")
print(target_os_perf.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]

for ax, col, label, color in [
    (axes[0], "median_ctr", "Median CTR", colors[0]),
    (axes[1], "median_cvr", "Median CVR", colors[1]),
    (axes[2], "median_roas", "Median ROAS", colors[2]),
]:
    ax.bar(
        target_os_perf["target_os"], target_os_perf[col], color=color, edgecolor="white", width=0.5
    )
    ax.set_title(f"{label} by Target OS Setting", fontweight="bold")
    ax.set_ylabel(label)

plt.suptitle("Does Targeting a Specific OS Improve Performance?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Daily Impressions Trend Over Time

In [ ]:
daily_trend = daily.groupby(["date", "os"])["impressions"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
for os_val, color in [("Android", "#3ddc84"), ("iOS", "#555555")]:
    sub = daily_trend[daily_trend["os"] == os_val]
    ax.plot(sub["date"], sub["impressions"] / 1e6, label=os_val, color=color, linewidth=2)

ax.set_title("Daily Impressions Over Time — Android vs iOS", fontweight="bold")
ax.set_ylabel("Impressions (millions)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()